# Sourav's Wedding Gallery - SOTA Face Processor (v3)

This version uses State-of-the-Art (SOTA) models for the best possible results:
- **Detector**: `RetinaFace` (Industry leading accuracy for finding faces, even if they are partially covered or at weird angles).
- **Recognizer**: `Facenet512` or `ArcFace` (Highly precise facial embeddings used in professional security systems).
- **Clustering**: `AgglomerativeClustering` (Better than DBSCAN for grouping people correctly).

---

### Step 1: Set up GPU and Install SOTA Libraries
Go to **Runtime** > **Change runtime type** and ensure **T4 GPU** is selected.

In [ ]:
!pip install deepface tf-keras requests numpy Pillow scikit-learn

### Step 2: Upload `manifest.json` from your `public/` folder.

In [ ]:
from google.colab import files
import os

uploaded = files.upload()

if 'manifest.json' not in uploaded:
    print("❌ Please upload 'manifest.json'!")
else:
    print("✅ manifest.json uploaded successfully.")

### Step 3: Run SOTA Processing
This will take longer than previous versions because the models are much more complex, but the results will be far superior.

In [ ]:
import json
import requests
import numpy as np
from PIL import Image
from deepface import DeepFace
from sklearn.cluster import AgglomerativeClustering
import shutil
import os

# Settings
DETECTOR_BACKEND = 'retinaface' # Options: 'retinaface', 'mtcnn', 'opencv', 'ssd', 'yolov8'
MODEL_NAME = 'Facenet512' # Options: 'VGG-Face', 'Facenet512', 'ArcFace'
DISTANCE_METRIC = 'cosine'
SIMILARITY_THRESHOLD = 0.4 # Lower = stricter grouping. Try 0.3 to 0.5.

MANIFEST_PATH = "manifest.json"
OUTPUT_FACES_PATH = "faces.json"
THUMBNAILS_DIR = "face-thumbnails"
IMAGES_DIR = "downloaded_images"

if not os.path.exists(THUMBNAILS_DIR):
    os.makedirs(THUMBNAILS_DIR)
if not os.path.exists(IMAGES_DIR):
    os.makedirs(IMAGES_DIR)

with open(MANIFEST_PATH, 'r') as f:
    manifest = json.load(f)

known_encodings = []
image_data = []

print(f"Processing {len(manifest)} images with SOTA models...")

for i, item in enumerate(manifest):
    public_id = item['public_id']
    url = item['url']
    filename = public_id.replace('/', '_') + ".jpg"
    filepath = os.path.join(IMAGES_DIR, filename)

    # Download if not exists
    if not os.path.exists(filepath):
        try:
            r = requests.get(url, timeout=15)
            if r.status_code == 200:
                with open(filepath, 'wb') as f:
                    f.write(r.content)
        except:
            continue

    if os.path.exists(filepath):
        try:
            # SOTA detection and representation
            results = DeepFace.represent(
                img_path = filepath, 
                model_name = MODEL_NAME, 
                detector_backend = DETECTOR_BACKEND,
                enforce_detection = False,
                align = True
            )

            for res in results:
                # Only include if confidence is reasonable (if enforce_detection is False)
                if res.get('face_confidence', 0) > 0.6:
                    known_encodings.append(res['embedding'])
                    image_data.append({
                        "public_id": public_id, 
                        "box": res['facial_area'], # x, y, w, h
                        "filepath": filepath
                    })
            
            if i % 10 == 0:
                print(f"Progress: {i}/{len(manifest)} images. Found {len(known_encodings)} faces so far...")
        except Exception as e:
            print(f"  - Skip {public_id} due to error")

if known_encodings:
    print(f"\nTotal faces detected: {len(known_encodings)}. Grouping identities...")
    
    # Convert to numpy array
    X = np.array(known_encodings)
    
    # Agglomerative Clustering is more reliable for identities
    # distance_threshold is used instead of n_clusters
    clt = AgglomerativeClustering(
        n_clusters=None, 
        distance_threshold=SIMILARITY_THRESHOLD, 
        metric=DISTANCE_METRIC, 
        linkage='average'
    )
    labels = clt.fit_predict(X)
    label_ids = np.unique(labels)

    faces_result = []
    for label_id in label_ids:
        indexes = np.where(labels == label_id)[0]
        if len(indexes) < 2: continue # Filter out one-off detections to reduce clutter
        
        rep_face = image_data[indexes[0]]
        
        img = Image.open(rep_face['filepath'])
        box = rep_face['box']
        x, y, w, h = box['x'], box['y'], box['w'], box['h']
        
        # Add padding
        pad_w, pad_h = w // 2, h // 2
        face_img = img.crop((
            max(0, x - pad_w), 
            max(0, y - pad_h), 
            min(img.width, x + w + pad_w), 
            min(img.height, y + h + pad_h)
        ))
        face_img.thumbnail((200, 200))
        
        thumb_name = f"face_{label_id}.jpg"
        face_img.save(os.path.join(THUMBNAILS_DIR, thumb_name), "JPEG", quality=90)
        
        photos = sorted(list(set([image_data[idx]['public_id'] for idx in indexes])))
        faces_result.append({
            "id": int(label_id), 
            "name": f"Person {label_id+1}", 
            "thumbnail": f"/face-thumbnails/{thumb_name}", 
            "photos": photos,
            "count": len(photos)
        })

    faces_result.sort(key=lambda x: x['count'], reverse=True)
    
    with open(OUTPUT_FACES_PATH, 'w') as f:
        json.dump(faces_result, f, indent=2)
    
    print(f"\n✅ Found {len(faces_result)} unique people!")
    print(f"✅ Results saved to {OUTPUT_FACES_PATH}")
else:
    print("❌ No faces found. Check if the manifest URLs are correct.")

### Step 4: Download Results
Download the `gallery_faces.zip` file, extract it, and place the contents in your project's `public/` folder.

In [ ]:
import shutil
from google.colab import files

if os.path.exists(OUTPUT_FACES_PATH):
    os.makedirs("results", exist_ok=True)
    shutil.copy(OUTPUT_FACES_PATH, "results/faces.json")
    if os.path.exists("results/face-thumbnails"):
        shutil.rmtree("results/face-thumbnails")
    shutil.copytree(THUMBNAILS_DIR, "results/face-thumbnails")
    
    shutil.make_archive("gallery_faces", 'zip', "results")
    files.download("gallery_faces.zip")
    print("✅ Zip file created and download started!")
else:
    print("❌ Nothing to download.")